# 01 — Dataset Overview

Reproducible descriptive statistics for the **Scientific Data** Data Descriptor.

**Manuscript sections:** Background & Summary · Data Records

**Outputs:** tables and figures saved under `notebooks/data_paper/figures/`.

In [1]:
from pathlib import Path

import altair as alt
import polars as pl

from config.paths import REPO_ROOT, SILVER_FACE_CLEAN, SILVER_PROCESSOS

FIGURES_DIR = REPO_ROOT / "projects/litigancia/notebooks/data_paper/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

lf_processos = pl.scan_delta(str(SILVER_PROCESSOS))
lf_face = pl.scan_delta(str(SILVER_FACE_CLEAN))


## 1. Scale and coverage

In [2]:
overview = lf_processos.select(
    pl.len().alias("n_decisoes"),
    pl.col("id_processo").n_unique().alias("n_processos_unicos"),
    pl.col("cd_processo").n_unique().alias("n_cd_unicos"),
    pl.col("comarca").n_unique().alias("n_comarcas"),
    pl.col("foro").n_unique().alias("n_foros"),
    (pl.len() / pl.col("id_processo").n_unique()).alias("decisoes_por_processo_media"),
).collect()

face_overview = lf_face.select(
    pl.len().alias("n_face_rows"),
    pl.col("cd_processo").n_unique().alias("n_face_processos_unicos"),
).collect()

print("=== processos_delta ===")
print(overview)
print("\n=== face_processos_clean_delta ===")
print(face_overview)

overview.write_csv(FIGURES_DIR / "table_scale_overview.csv")
face_overview.write_csv(FIGURES_DIR / "table_face_overview.csv")

=== processos_delta ===
shape: (1, 6)
┌────────────┬────────────────────┬─────────────┬────────────┬─────────┬───────────────────────────┐
│ n_decisoes ┆ n_processos_unicos ┆ n_cd_unicos ┆ n_comarcas ┆ n_foros ┆ decisoes_por_processo_med │
│ ---        ┆ ---                ┆ ---         ┆ ---        ┆ ---     ┆ ia                        │
│ u32        ┆ u32                ┆ u32         ┆ u32        ┆ u32     ┆ ---                       │
│            ┆                    ┆             ┆            ┆         ┆ f64                       │
╞════════════╪════════════════════╪═════════════╪════════════╪═════════╪═══════════════════════════╡
│ 6100561    ┆ 5812126            ┆ 5812127     ┆ 321        ┆ 339     ┆ 1.049626                  │
└────────────┴────────────────────┴─────────────┴────────────┴─────────┴───────────────────────────┘

=== face_processos_clean_delta ===
shape: (1, 2)
┌─────────────┬─────────────────────────┐
│ n_face_rows ┆ n_face_processos_unicos │
│ ---         ┆ --- 

## 2. Subject and court distributions

In [3]:
def top_counts(lf: pl.LazyFrame, column: str, n: int = 15) -> pl.DataFrame:
    return (
        lf.group_by(column)
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
        .head(n)
        .collect()
    )

top_assunto = top_counts(lf_processos, "assunto")
top_comarca = top_counts(lf_processos, "comarca")
top_foro = top_counts(lf_processos, "foro")
top_vara = top_counts(lf_processos, "vara")

for name, df in [
    ("assunto", top_assunto),
    ("comarca", top_comarca),
    ("foro", top_foro),
    ("vara", top_vara),
]:
    print(f"\n=== Top {name} ===")
    display(df)
    df.write_csv(FIGURES_DIR / f"table_top_{name}.csv")


=== Top assunto ===


assunto,n
str,u32
"""IPTU/ Imposto Predial e Territ…",2092880
"""Dívida Ativa""",1720485
"""Assunto não informado.""",653527
"""ISS/ Imposto sobre Serviços""",351661
"""ICMS/ Imposto sobre Circulação…",202801
…,…
"""Multas e demais Sanções""",81215
"""Taxas""",47485
"""Taxa de Coleta de Lixo""",33648



=== Top comarca ===


comarca,n
str,u32
"""SÃO PAULO""",384810
"""Guarulhos""",254029
"""Campinas""",182904
"""Praia Grande""",156067
"""Osasco""",148328
…,…
"""Piracicaba""",86912
"""Guarujá""",84574
"""São José do Rio Preto""",77364



=== Top foro ===


foro,n
str,u32
"""Foro das Execuções Fiscais Mun…",306917
"""Foro de Guarulhos""",254029
"""Foro de Campinas""",182904
"""Foro de Praia Grande""",156067
"""Foro de Osasco""",148328
…,…
"""Foro de Piracicaba""",86912
"""Foro de Guarujá""",84574
"""Foro de São José do Rio Preto""",77364



=== Top vara ===


vara,n
str,u32
"""SAF - Serviço de Anexo Fiscal""",1307050
"""Vara da Fazenda Pública""",961182
"""SEF - Setor de Execuções Fisca…",810625
"""Setor das Execuções Fiscais""",433701
"""Vara Única""",365926
…,…
"""SETOR DE EXECUÇÕES FISCAIS DA …",126544
"""2ª Vara""",115626
"""SEF - Setor das Execuções Fisc…",100471


## 3. Temporal coverage

In [4]:
lf_time = lf_processos.with_columns(
    pl.col("data_disponibilizacao")
    .str.strptime(pl.Date, "%d/%m/%Y", strict=False)
    .alias("dt_pub")
)

serie_mensal = (
    lf_time.filter(pl.col("dt_pub").is_not_null())
    .group_by(pl.col("dt_pub").dt.truncate("1mo").alias("mes"))
    .agg(pl.len().alias("n_decisoes"))
    .sort("mes")
    .collect()
)

por_ano = (
    lf_time.filter(pl.col("dt_pub").is_not_null())
    .group_by(pl.col("dt_pub").dt.year().alias("ano"))
    .agg(pl.len().alias("n"))
    .sort("ano")
    .collect()
)

serie_mensal.write_csv(FIGURES_DIR / "table_decisoes_por_mes.csv")
por_ano.write_csv(FIGURES_DIR / "table_decisoes_por_ano.csv")

chart_mes = (
    serie_mensal.plot.line(x="mes", y="n_decisoes")
    .properties(title="Decisões publicadas por mês", width=800, height=400)
)
chart_mes.save(str(FIGURES_DIR / "fig_decisoes_por_mes.png"), scale_factor=2)
display(chart_mes)

chart_ano = (
    por_ano.plot.bar(
        x=alt.X("ano:Q", axis=alt.Axis(format="d", tickMinStep=1), title="Ano"),
        y="n",
    )
    .properties(title="Decisões publicadas por ano", width=800, height=400)
)
chart_ano.save(str(FIGURES_DIR / "fig_decisoes_por_ano.png"), scale_factor=2)
display(chart_ano)

alt.Chart(...)

alt.Chart(...)

## 4. FACE table summary (financial and sentence metadata)

In [5]:
ANO_MIN = 2000
TETO_VALOR = 1_000_000_000

lf_face_clean = (
    lf_face
    .filter(pl.col("distribuicao_data").dt.year() >= ANO_MIN)
    .filter(
        pl.col("valor_corrigido_atual").is_null()
        | (pl.col("valor_corrigido_atual") <= TETO_VALOR)
    )
)

tramitacao_stats = (
    lf_face_clean.select(pl.col("tempo_tramitacao_meses").drop_nulls())
    .collect()["tempo_tramitacao_meses"]
    .describe()
)
print(tramitacao_stats)
tramitacao_stats.write_csv(FIGURES_DIR / "table_tramitacao_stats.csv")

top_sentenca = (
    lf_face_clean.group_by("tipo_sentença")
    .len()
    .sort("len", descending=True)
    .head(15)
    .collect()
)
display(top_sentenca)
top_sentenca.write_csv(FIGURES_DIR / "table_top_tipo_sentenca.csv")

shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ value      │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 5.556555e6 │
│ null_count ┆ 0.0        │
│ mean       ┆ 67.156719  │
│ std        ┆ 64.423838  │
│ min        ┆ -455.81    │
│ 25%        ┆ 17.71      │
│ 50%        ┆ 45.5       │
│ 75%        ┆ 100.92     │
│ max        ┆ 317.21     │
└────────────┴────────────┘


tipo_sentença,len
str,u32
"""[""Extinta a Execução/Cumprimen…",2398582
"""[""Extinto o Processo sem Resol…",445356
"""[""Declarada Decadência ou Pres…",415522
"""[""Extinto o Processo sem Resol…",324400
"""[""Extinta a Punibilidade por P…",219986
…,…
"""[""Extinta a Execução pela Pres…",87253
"""[null]""",82677
"""[""Indeferida a Petição Inicial…",71816
